# ESA-ADB Lite: LOF

Local Outlier Factor; heavy on the full ESA test split.

> This algorithm can be slow on `target` channels. Start with `subset` and small splits locally; use AutoDL for larger runs.

This notebook is a thin experiment launcher. The actual implementation lives in `algorithms.py` and `run_benchmark.py`, so notebook runs and command-line runs stay consistent.


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

REPO = Path.cwd()
if not (REPO / "run_benchmark.py").exists():
    REPO = REPO.parent

ALGORITHM = "LOF"
DATA_PATH = REPO.parent / "data" / "preprocessed"
OUTPUT_DIR = REPO / "results" / "notebooks"

# Good local defaults. For larger runs, change these before running.
DATASETS = ["3_months"]
CHANNEL_GROUPS = ["subset"]
INCLUDE_VUS = False
VUS_MAX_POINTS = 50000
SKIP_OFFICIAL_METRICS = True


## Run

Adjust `DATASETS`, `CHANNEL_GROUPS`, and `INCLUDE_VUS` above, then run this cell.


In [ ]:
cmd = [
    sys.executable,
    "run_benchmark.py",
    str(DATA_PATH),
    "--preset",
    "extended",
    "--output-dir",
    str(OUTPUT_DIR),
    "--datasets",
    *DATASETS,
    "--channel-groups",
    *CHANNEL_GROUPS,
    "--algorithms",
    ALGORITHM,
]

if SKIP_OFFICIAL_METRICS:
    cmd.append("--skip-official-metrics")
if INCLUDE_VUS:
    cmd.extend([
        "--include-vus-metrics",
        "--vus-max-points",
        str(VUS_MAX_POINTS),
        "--vus-max-buffer",
        "50",
        "--vus-max-thresholds",
        "30",
    ])

print(" ".join(str(x) for x in cmd))
subprocess.run(cmd, cwd=REPO, check=True)


## Inspect Latest Result

This loads the latest notebook run for this algorithm.


In [ ]:
run_dirs = sorted((OUTPUT_DIR / "extended").glob("*"))
if not run_dirs:
    raise FileNotFoundError("No notebook run found.")

latest = run_dirs[-1]
print(latest)
df = pd.read_csv(latest / "results.csv")
df


In [ ]:
summary_path = latest / "summary_by_algorithm.csv"
if summary_path.exists():
    pd.read_csv(summary_path)
